## Transition Recursive equation of Local Linear SDE

From locally linear approximation method, we evaluated the drift coefficient $A$ and the control vector $\alpha{}$ using neural net.

The key strength of linear approximation is that the SDE could be solved analytically without simulation - namely, simulation-free property.

Therefore, the average and covariance of the normal distribution ( local linear approxmaiton is a type of OU process, and the closed solution distribution is given; $X_{t} \sim \mathcal{N}(\mu{}_t,\Sigma{}_t)$ is calculated directly from the transition recursive equation.

Simulation-free property gives rise to the independent gaussian sampling at each time stamp of the given process.




### linear SDE coefficient

In [7]:
import torch
from torch import nn

from src import make_ou_data
from src import ObsEncoder
from src import ACSSMTransformer, masking_unseen_data_Full_Assim
from src import PSDLocalLinearApprox

# 1. Data preparation
data = make_ou_data(32, 40, 3)
obs = data["observation"]
obs_mask = data["obs_mask"]
times = data["times"]

# 2. Encoding the observation
encoder = ObsEncoder(obs_dim= obs.shape[-1], embed_dim=8)
sampled_latent_obs = encoder(obs, obs_mask)
B,T,D =sampled_latent_obs.shape # (32, 40, 8)

# 2.1 Transformer layer
transformer = ACSSMTransformer(D,2)
z_raw = transformer(sampled_latent_obs, times, obs_mask)
z = masking_unseen_data_Full_Assim(z_raw, obs_mask)

# 3. Local linear approximation
localLinApp = PSDLocalLinearApprox(D,D)
linear_SDE_param = localLinApp(z)




time_embedding: torch.Size([32, 40, 8])
transformer_input: torch.Size([32, 40, 8])
z_raw: torch.Size([32, 40, 8])


### Change basis and one-step integration

The original LSDE is
$$
    dX_t = (-A_iX_t + \alpha{})dt + dW_t
$$

change basis using $E$

$$
    \hat{X}_t = E^{\intercal}X_t, \quad{} \hat{\alpha{}}_i = E^{\intercal}\alpha{}_i, \quad{} A_i \to D_i=\text{diag}(\lambda{}_i)
$$

Therfore, SDE turns into simple 1-d SDE
$$
    d\hat{X}_{t} = \left( -\text{diag}(\lambda{}_i)\hat{X}_t + \hat{\alpha{}}_i \right)dt + d\hat{W}_t
$$

We can integrate this by multiplying $e^{\lambda{}t}$ to both hand side

$$
    \hat{X}_{t+\Delta{}} = e^{-\lambda{}\Delta{}}\hat{X}_t 
    + \hat{\alpha{}}\int_{0}^{\Delta{}} e^{-\lambda{}u}\,\mathrm{d}u 
    + \int_{t}^{t+\Delta{}} e^{-\lambda{}(t+\Delta{}-s)}\,\mathrm{d}W_s 
$$

We define 3 coefficient here

$$
    f = e^{-\lambda{}\Delta{}}, \quad{} g = \int_{0}^{\Delta{}} e^{-\lambda{}u}\,\mathrm{d}u = \frac{1-e^{-\lambda{}\Delta{}}}{\lambda{}}\\[1em]
    \eta{} = \int_{t}^{t+\Delta{}} e^{-\lambda{}(t+\Delta{}-s)}\,\mathrm{d}W_s = \mathcal{N}(0,q)
    , \quad{}\text{where } q = \text{Var}(\eta{})
$$

$\eta$ is the summation of independent Wiener increment, which follows gaussian distribution with zero mean.
from Ito isometry, you can calculate the variance :

$$
    \text{Var}(\eta) = q = \int_{t}^{t+\Delta{}} e^{-2\lambda{}(t+\Delta{}-s)}\,\mathrm{d}s \\[1em]
    = \int_{0}^{\Delta{}} e^{-2\lambda{}s}\,\mathrm{d}s = \frac{1-e^{-2\lambda{}\Delta{}}}{2\lambda{}}
$$

Wrapping up, in vector notation,

$$
    \hat{X}_{t+\Delta{}} = \hat{f} \odot \hat{X}_{t} + \hat{g} \odot \hat{\alpha{}} + \hat{q}
$$

Where, 
$$
    \hat{f}_{i} = e^{-\lambda{}_{i}\Delta{}},\quad{} \hat{g}_{i} = \frac{1-e^{-\lambda{}_{i}\Delta{}}}{\lambda{}_{i}}, \quad{} \hat{q}_{i} = \frac{1-e^{-2\lambda{}_{i}\Delta{}}}{2\lambda{}_{i}}
$$



if the $\lambda{}$ is very small, the function $\frac{1-e^{-\lambda{}\Delta{}}}{\lambda{}}$ would not be numerically stable.

Therefore, define $\phi{}$ according to the magnitude of $x$.

$$
    \begin{cases}
        \phi{}(x) &= \frac{1-e^{-x}}{x} \approx 1- \frac{x}{2} + \frac{x^{2}}{6} - \frac{x^{3}}{24} , &x \approx 0 \\[1em]
        \phi{}(x) &= \frac{1-e^{-x}}{x}, &x > 0
    \end{cases}
    
$$


In [ ]:
def phi(x: torch.Tensor) -> torch.Tensor:
    """
    phi(x) = (1 - exp(-x)) / x
    phi(0) = 1

    Input / output: same shape
    """
    small = x.abs() < 1e-4

    # Regular expression is for not small lambda.
    x_regular = torch.where(
        small,
        torch.ones_like(x),
        x,
    )
    regular = -torch.expm1(-x_regular) / x_regular

    # Taylor expand of phi function for small lambda 
    x_small = torch.where(
        small,
        x,
        torch.zeros_like(x),
    )
    series = (
        1.0
        - x_small / 2.0
        + x_small.square() / 6.0
        - x_small**3 / 24.0
    )

    return torch.where(small, series, regular)

### Calculate average and covarinace of X_t

In the previous cell, we derived

$$
    \hat{X}_{t+\Delta{}} = \hat{f} \odot \hat{X}_{t} + \hat{g} \odot \hat{\alpha{}} + \hat{q}
$$

from this equation, we can easily derive the clsoed recurrence form of the average and covariance for the distribution of X, namely,

$$
\begin{align*}
    &\hat{m}_{t+\Delta{}} = F \hat{m}_t + G \\
    &\hat{\Sigma{}}_{t+\Delta{}} = F\hat{\Sigma{}}_{t}F^{\intercal} + Q ,\quad{} \text{where, } \hat{X}_{t} \sim \mathcal{N}(\hat{m}_t, \hat{\Sigma{}}_{t})
\end{align*}
$$

$F,G,Q$ is diagonal matrix whose diagonal element is $f,g,q$ vector





In [10]:
def transition_coefficients(
    local_params: dict[str, torch.Tensor],
    times: torch.Tensor,
) -> dict[str, torch.Tensor]:
    """
    Convention:
        parameters at index i apply on [times[i], times[i+1])

    Assumes:
        dX = (-A X + alpha) dt + dW
        A_i = E diag(lambda_i) E.T
        E is shared across all batches and intervals.

    All returned tensors:
        (B, T-1, d)
    """
    eigen_vals = local_params["eigen_vals"]  # (B, T, d)
    alpha_all = local_params["alpha"]       # (B, T, d)
    E = local_params["E"]                   # (d, d)

    times = times.to(
        device=eigen_vals.device,
        dtype=eigen_vals.dtype,
    )



    # 1. time interval definition
    """
        (t_1,t_2, ..., t_T-1)
       -(t_0,t_1, ..., t_T-2)
       =(dt0,dt1, ..., dt_T-2)

        dt_i = [ t_i, t_i+1 )
        A_i,alpha_i  <->  dt_i

        => Total T-1 intervals
    """
    lam = eigen_vals[:, :-1, :]             # (B, T-1, d)
    alpha = alpha_all[:, :-1, :]            # (B, T-1, d)

    dt = (
        times[:, 1:] - times[:, :-1]
    ).unsqueeze(-1)                         # (B, T-1, 1)

    # 2. Change basis from original  to E ( eigen basis of A matrix )
    alpha_hat = alpha @ E                   # (B, T-1, d)

    # 3. definiiton for convinience
    x = lam * dt                            # (B, T-1, d)


    # 4. evaluate the coefficient. 
    decay = torch.exp(-x)       #  function f
    control_gain = dt * phi(x)  #  function g
    noise_var = dt * phi(2.0 * x)

    
    # 5. Affine coefficient for average and covariance of X_t
    shift_hat = control_gain * alpha_hat
    var_decay = decay.square()


    return {
        "decay": decay,
        "control_gain": control_gain,
        "shift_hat": shift_hat,
        "noise_var": noise_var,
        "var_decay": var_decay,
    }

## Parallel Scan algorithm ( Hillis-Steele scan algorithm )

The original paper used Blelloch scan in "Algorithm 2" in the paper. Here, however, we use more direct approach for parallel scan, which is Hillis-Steele scan.

you can easily check that this relation is affine transform. and it is well knwon that this transformaiton is associative.

The corresponding binary operator to the affine transform is $(F,G) \oplus (A,B) = (FA , FB + G)$, you can check that $\oplus$ is associative operator.

Because the average and covariance relation is associative, we can use parallel scan algorithm to get the whole spectrum of $\{\hat{m}_t, \hat{\Sigma{}}_t\}_t$.

Instead manually evaluating the value from the previous step, using associative property we can evaluate the following calculation in parallel way.
